# **Main Quest**

**Transformer 모델과 비교하여 변경이 필요한 부분**

1. Decoder 구조만 사용

    *   기존 Transformer 모델의 Encoder-Decoder 구조에서 Encoder 부분 삭제
    *   Decoder의 self-attention만 사용


2.   Causal Mask 적용

      *   미래의 토큰을 참조하지 못하도록 하기 위한 목적


3.   입력과 출력 구조 변경

      *   Encoder 삭제로 인한 encoder_input 제거
      *   하나의 토큰 시퀀스로만 입력으로 받아 다음 토큰을 예측


4.   Positional Encoding 변경

      *   관련 논문 "Improving Language Understanding by Generative Pre-Training"(https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)을 참조하여 Learned positional embedding 적용(기존은 사인/코사인 기반 위치 인코딩 방식)


5.   출력층 변경

      *   각 time-step에서 vocabulary 전체에 대해 softmax 수행





In [1]:
# 데이터셋 다운로드 및 압축 해제

import os
import zipfile
import urllib.request

url = 'http://www.cs.cornell.edu/~cristian/data/cornell_movie_dialogs_corpus.zip'
zip_filename = 'cornell_movie_dialogs.zip'

if not os.path.exists(zip_filename):
    urllib.request.urlretrieve(url, zip_filename)

with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall('cornell')

print("데이터셋 준비")

데이터셋 준비


In [3]:
# GPT-1 pretraining용 Language Model 데이터 구성

    # 하나의 문장 시퀀스로 다음 토큰을 예측하도록 구성

import re

lines_path = "cornell/cornell movie-dialogs corpus/movie_lines.txt"

def load_lines(path):
    lines = {}
    with open(path, encoding='iso-8859-1') as f:
        for line in f:
            parts = line.strip().split(" +++$+++ ")
            lines[parts[0]] = parts[-1]
    return list(lines.values())

    # 텍스트 정제 및 전처리(특수문자 제거 및 단어 수 3개 이하 문장 제거)

sentences = load_lines(lines_path)

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9?.!,']", " ", text)
    return text

sentences = [clean_text(s) for s in sentences if len(s.split()) > 3]
print("Sample:", sentences[0])

Sample: okay    you're gonna need to learn how to lie.


In [4]:
# 토크나이징 및 GPT Input 구성

    # GPT 논문에 기반한 토큰+위치 임베딩 방식 적용
    # GPT 논문에 따라 input = x[:-1], label = x[1:] 적용

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

VOCAB_SIZE = 8000   # Vocab size : 8,000 적용
MAX_LEN = 40        # 패딩 : 길이 40으로 고정

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<unk>")      # 정수 인코딩
tokenizer.fit_on_texts(sentences)

sequences = tokenizer.texts_to_sequences(sentences)
sequences = pad_sequences(sequences, maxlen=MAX_LEN, padding='post')

inputs = sequences[:, :-1]
targets = sequences[:, 1:]

print("Input shape:", inputs.shape)
print("Target shape:", targets.shape)

Input shape: (233358, 39)
Target shape: (233358, 39)


In [9]:
# GPT-1 한 개 블록(Transformer Decoder Block) 정의

import tensorflow.keras.layers as layers
from tensorflow.keras.models import Model

class GPTBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.att = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="gelu"),
            layers.Dense(embed_dim),
        ])
        self.ln1 = layers.LayerNormalization()
        self.ln2 = layers.LayerNormalization()

    def call(self, x):
        seq_len = tf.shape(x)[1]

        # Causal mask 직접 생성

        causal_mask = tf.linalg.band_part(
            tf.ones((seq_len, seq_len)), -1, 0
        )

        attn_output = self.att(
            x, x, attention_mask=causal_mask
        )
        x = self.ln1(x + attn_output)
        ffn_output = self.ffn(x)
        return self.ln2(x + ffn_output)


In [10]:
# GPT-1 전체 모델 구성

EMBED_DIM = 128
NUM_HEADS = 4
FF_DIM = 512
NUM_LAYERS = 4

inputs_ids = layers.Input(shape=(MAX_LEN-1,))

    # 토큰 + Learned Position Embedding(위치를 임베딩을 학습하는 방식)

token_emb = layers.Embedding(VOCAB_SIZE, EMBED_DIM)(inputs_ids)
pos_emb = layers.Embedding(MAX_LEN, EMBED_DIM)(
    tf.range(start=0, limit=MAX_LEN-1)
)

x = token_emb + pos_emb

for _ in range(NUM_LAYERS):
    x = GPTBlock(EMBED_DIM, NUM_HEADS, FF_DIM)(x)

outputs = layers.Dense(VOCAB_SIZE, activation="softmax")(x)

model = Model(inputs=inputs_ids, outputs=outputs)
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 39)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_4 (Embedding)         │ (None, 39, 128)        │     1,024,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ add_2 (Add)                     │ (None, 39, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gpt_block_2 (GPTBlock)          │ (None, 39, 128)        │       396,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gpt_block_3 (GPTBlock)          │ (None, 39, 128)        │       396,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gpt_block_4 (GPTBlock)          │ (None, 39, 128)        │       396,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gpt_block_5 (GPTBlock)          │ (None, 39, 128)        │       396,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 39, 8000)       │     1,032,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,640,128 (13.89 MB)

 Trainable params: 3,640,128 (13.89 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# 모델 학습

history = model.fit(
    inputs,
    targets,
    batch_size=64,
    epochs=30,
    validation_split=0.1
)

Epoch 1/30
3282/3282 ━━━━━━━━━━━━━━━━━━━━ 129s 34ms/step - accuracy: 0.7227 - loss: 1.9656 - val_accuracy: 0.7399 - val_loss: 1.5240
Epoch 2/30
3282/3282 ━━━━━━━━━━━━━━━━━━━━ 101s 31ms/step - accuracy: 0.7475 - loss: 1.4533 - val_accuracy: 0.7434 - val_loss: 1.4853
Epoch 3/30
3282/3282 ━━━━━━━━━━━━━━━━━━━━ 100s 30ms/step - accuracy: 0.7511 - loss: 1.4007 - val_accuracy: 0.7452 - val_loss: 1.4687
Epoch 4/30
3282/3282 ━━━━━━━━━━━━━━━━━━━━ 101s 31ms/step - accuracy: 0.7533 - loss: 1.3700 - val_accuracy: 0.7460 - val_loss: 1.4624
Epoch 5/30
3282/3282 ━━━━━━━━━━━━━━━━━━━━ 98s 30ms/step - accuracy: 0.7563 - loss: 1.3387 - val_accuracy: 0.7462 - val_loss: 1.4594
Epoch 6/30
3282/3282 ━━━━━━━━━━━━━━━━━━━━ 100s 31ms/step - accuracy: 0.7561 - loss: 1.3276 - val_accuracy: 0.7464 - val_loss: 1.4601
Epoch 7/30
3282/3282 ━━━━━━━━━━━━━━━━━━━━ 99s 30ms/step - accuracy: 0.7576 - loss: 1.3101 - val_accuracy: 0.7464 - val_loss: 1.4655
Epoch 8/30
3282/3282 ━━━━━━━━━━━━━━━━━━━━ 100s 31ms/step - accuracy: 0.

In [12]:
# 입력에 따른 출력 생성 테스트

import numpy as np

def generate_text(prompt, max_new_tokens=20):
    tokens = tokenizer.texts_to_sequences([clean_text(prompt)])[0]
    tokens = tokens[-(MAX_LEN-1):]

    for _ in range(max_new_tokens):
        padded = pad_sequences([tokens], maxlen=MAX_LEN-1)
        preds = model.predict(padded, verbose=0)
        next_token = np.argmax(preds[0, len(tokens)-1])
        tokens.append(next_token)

    return tokenizer.sequences_to_texts([tokens])[0]

print(generate_text("i love you"))

i love you a a a a a <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> a


**결과 분석 보고서**

*   Loss의 감소와 Accuracy의 상승 추세를 보아 모델이 정상적으로 학습되었음을 확인
*   입력을 통한 완벽한 출력 모델을 생성하지 못하였으나 이는 작은 데이터셋과 얕은 모델(레이어 수 등)의 원인으로 판단됨(모델 성능 보다는 GPT의 구조 이해에 의의를 둠)
*   논문에서 지적하는 한계점으로 GPT-1의 많은 연산량과 일반화 문제는 개선될 필요가 있음


**회고**

*   디코더 기반 구조의 GPT-1 모델을 실제로 구현함으로써 논문을 읽는 것보다 더 이해가 수월하였음
*   이번 메인 퀘스트를 통하여 과제의 달성을 떠나서 GPT 초기 모델의 구조를 이해하고 무엇을 보완하면서 발전해왔는지 알게된 기회가 되었음

